# 07 — End-to-end production ML project

**Estimated time:** 150–210 minutes  
**Prerequisites:** notebooks 00–06.  
**Depends on:** every prior data, evaluation, leakage, explanation, and monitoring decision.

## Learning objectives

- Rebuild a clean train-to-inference pipeline with a data contract.
- Select a threshold on validation, then evaluate the sealed test exactly once.
- Serialize, reload, and verify deterministic batch inference.
- Produce reproducibility metadata, checks, a model card, and monitoring plan.


In [1]:
import hashlib, json, os, platform, random, warnings
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
# Known interoperability/UI warnings do not affect predictions or notebook execution.
warnings.filterwarnings("ignore", message="X does not have valid feature names, but LGBMClassifier")
warnings.filterwarnings("ignore", message="IProgress not found.*")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (average_precision_score, balanced_accuracy_score,
                             brier_score_loss, confusion_matrix, f1_score, log_loss,
                             precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
TARGET = "y"
LEAKAGE_COLUMNS = ["duration"]

def project_root():
    '''Return the course root when present, otherwise the notebook directory.'''
    # Return the course root when present, otherwise the notebook's directory.
    return ROOT

def set_seed(seed=SEED):
    '''Seed Python and NumPy RNGs for reproducible notebook runs.'''
    random.seed(seed)
    np.random.seed(seed)

def fast_mode():
    '''Report whether notebooks should use the reduced fast-mode sample.'''
    # Set FAST_MODE=0 for full-size experiments; laptop mode is the default.
    return os.getenv("FAST_MODE", "1").lower() not in {"0", "false", "no"}

def bank_data_path():
    '''Locate the bundled Bank Marketing CSV file.'''
    # The course ships with a local dataset; notebooks never access the network.
    path = project_root() / "data" / "raw" / "bank-full.csv"
    if not path.is_file():
        raise FileNotFoundError(
            f"Expected the bundled Bank Marketing data at {path}. "
            "Run the notebook from the course root or place bank-full.csv there."
        )
    return path

def file_sha256(path):
    '''Compute the SHA-256 digest of a local file.'''
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def load_bank_data(include_duration=False):
    '''Load the Bank Marketing dataset and drop leakage columns by default.'''
    # Load the data, encode y, and exclude post-call duration by default.
    frame = pd.read_csv(bank_data_path(), sep=";")
    frame[TARGET] = frame[TARGET].map({"no": 0, "yes": 1}).astype("int8")
    if not include_duration:
        frame = frame.drop(columns=LEAKAGE_COLUMNS)
    return frame

def stratified_sample(frame, n, seed=SEED):
    '''Draw a label-preserving sample from a classified dataset.'''
    if n >= len(frame):
        return frame.copy()
    fractions = frame[TARGET].value_counts(normalize=True)
    counts = (fractions * n).round().astype(int)
    counts.iloc[0] += n - counts.sum()
    parts = [group.sample(n=min(counts.loc[label], len(group)),
                          random_state=seed + int(label))
             for label, group in frame.groupby(TARGET)]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

def make_splits(frame=None, reduced=None):
    '''Create deterministic stratified train, validation, and test splits.'''
    # Deterministic stratified 60/20/20 split; test stays sealed until notebook 07.
    from sklearn.model_selection import train_test_split
    frame = load_bank_data() if frame is None else frame
    train_val, test = train_test_split(
        frame, test_size=0.20, stratify=frame[TARGET], random_state=SEED)
    train, validation = train_test_split(
        train_val, test_size=0.25, stratify=train_val[TARGET], random_state=SEED)
    reduced = fast_mode() if reduced is None else reduced
    if reduced:
        train = stratified_sample(train, 12_000)
        validation = stratified_sample(validation, 4_000, SEED + 1)
        test = stratified_sample(test, 4_000, SEED + 2)
    return tuple(part.reset_index(drop=True) for part in (train, validation, test))

def split_xy(frame):
    '''Separate a frame into feature matrix and target vector.'''
    return frame.drop(columns=TARGET), frame[TARGET]

def feature_groups(frame):
    '''Identify numeric and categorical feature columns.'''
    features = frame.drop(columns=[TARGET], errors="ignore")
    categorical = features.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numerical = features.select_dtypes(include=np.number).columns.tolist()
    return numerical, categorical

def make_preprocessor(frame, scale_numeric=True):
    '''Build the preprocessing pipeline for numeric and categorical features.'''
    # Preprocessing is fitted only inside the enclosing training/CV pipeline.
    numerical, categorical = feature_groups(frame)
    numeric_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                 min_frequency=10, sparse_output=True)),
    ])
    return ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numerical),
        ("categorical", categorical_pipe, categorical),
    ], sparse_threshold=0.3)

def classification_metrics(y_true, probability, threshold=0.5):
    '''Compute ranking and threshold-based classification metrics.'''
    prediction = np.asarray(probability) >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {"roc_auc": roc_auc_score(y_true, probability),
            "average_precision": average_precision_score(y_true, probability),
            "log_loss": log_loss(y_true, probability),
            "brier_score": brier_score_loss(y_true, probability),
            "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
            "f1": f1_score(y_true, prediction, zero_division=0),
            "precision": precision_score(y_true, prediction, zero_division=0),
            "recall": recall_score(y_true, prediction, zero_division=0),
            "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "cost": float(fp + 5 * fn)}

def threshold_table(y_true, probability, thresholds=None):
    '''Evaluate classification metrics across a list of decision thresholds.'''
    thresholds = np.linspace(0.05, 0.80, 76) if thresholds is None else thresholds
    return pd.DataFrame([{"threshold": float(t),
                          **classification_metrics(y_true, probability, float(t))}
                         for t in thresholds])

def add_domain_features(frame):
    '''Create domain-inspired features for the Bank Marketing dataset.'''
    result = frame.copy()
    result["was_previously_contacted"] = (result["pdays"] != -1).astype("int8")
    result["pdays_clean"] = result["pdays"].replace(-1, np.nan)
    result["contact_pressure"] = result["campaign"] / (1 + result["previous"])
    result["balance_per_age"] = result["balance"] / result["age"].clip(lower=18)
    result["age_band"] = pd.cut(result["age"], bins=[0, 29, 39, 49, 59, np.inf],
                                labels=["<30", "30s", "40s", "50s", "60+"]).astype("object")
    return result.drop(columns=["pdays"])

def environment_metadata():
    '''Collect runtime metadata for experiment tracking.'''
    import sklearn
    return {"python": platform.python_version(), "platform": platform.platform(),
            "numpy": np.__version__, "pandas": pd.__version__,
            "scikit_learn": sklearn.__version__}

def write_json(data, path):
    '''Serialize structured data to a JSON file on disk.'''
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")

set_seed(SEED)
FAST_MODE = fast_mode()
CV_FOLDS = 3 if FAST_MODE else 5
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)
print({"FAST_MODE": FAST_MODE, "CV_FOLDS": CV_FOLDS, "seed": SEED})


{'FAST_MODE': True, 'CV_FOLDS': 3, 'seed': 42}


## Final protocol

We choose a regularized LightGBM pipeline as a pragmatic performance/complexity compromise. The
estimator and feature logic are fixed before test access. Development fits the model; validation
selects the cost-sensitive threshold; the unchanged model and threshold are then evaluated once on
test. We do not refit on development+validation before this evaluation because that would change the
calibrated operating behavior selected on validation.


In [2]:
import json, joblib, lightgbm as lgb
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

full = load_bank_data()
development, validation, test = make_splits(full, reduced=FAST_MODE)
X_dev, y_dev = split_xy(development)
X_val, y_val = split_xy(validation)
X_test, y_test = split_xy(test)  # first notebook that permits test labels

engineered_schema = add_domain_features(development)
final_pipeline = Pipeline([
    ("features", FunctionTransformer(add_domain_features, validate=False)),
    ("preprocess", make_preprocessor(engineered_schema, scale_numeric=False)),
    ("model", lgb.LGBMClassifier(
        n_estimators=280 if FAST_MODE else 500, learning_rate=.04, num_leaves=31,
        min_child_samples=40, subsample=.9, colsample_bytree=.9,
        reg_lambda=1.0, random_state=SEED, n_jobs=-1, verbosity=-1,
    )),
])
final_pipeline.fit(X_dev, y_dev)


,steps,"[('features', ...), ('preprocess', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,func,<function add...t 0x117353f60>
,inverse_func,None
,validate,False
,accept_sparse,False
,check_inverse,True
,feature_names_out,None
,kw_args,None


## Validation threshold selection; then one-time final test evaluation


In [3]:
validation_probability = final_pipeline.predict_proba(X_val)[:, 1]
threshold_result = threshold_table(y_val, validation_probability).sort_values("cost").iloc[0]
selected_threshold = float(threshold_result["threshold"])
print("frozen threshold:", selected_threshold)
display(pd.DataFrame({
    "validation": classification_metrics(y_val, validation_probability, selected_threshold)
}).T)

# FINAL TEST ACCESS: do not use these results to revise features, hyperparameters, or threshold.
test_probability = final_pipeline.predict_proba(X_test)[:, 1]
final_test_metrics = classification_metrics(y_test, test_probability, selected_threshold)
display(pd.DataFrame({"final_test": final_test_metrics}).T)


frozen threshold: 0.22000000000000003


,roc_auc,average_precision,log_loss,brier_score,balanced_accuracy,f1,precision,recall,specificity,cost
validation,0.791736,0.451605,0.287158,0.081743,0.725771,0.49075,0.450805,0.538462,0.91308,1387.0


,roc_auc,average_precision,log_loss,brier_score,balanced_accuracy,f1,precision,recall,specificity,cost
final_test,0.780306,0.422231,0.294007,0.083693,0.709874,0.472837,0.446768,0.502137,0.91761,1456.0


The final test estimate has sampling uncertainty and applies to this historical distribution. If
the result disappoints, it remains the honest result for this development cycle. Any subsequent
redesign starts a new cycle and needs fresh final-evaluation data.

## Data contract and batch inference


In [4]:
REQUIRED_COLUMNS = X_dev.columns.tolist()
NUMERIC_COLUMNS = X_dev.select_dtypes(include=np.number).columns.tolist()
CATEGORICAL_COLUMNS = X_dev.select_dtypes(include="object").columns.tolist()

def validate_batch(frame):
    if not isinstance(frame, pd.DataFrame):
        raise TypeError("batch must be a pandas DataFrame")
    missing = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
    extra = sorted(set(frame.columns) - set(REQUIRED_COLUMNS))
    if missing or extra:
        raise ValueError(f"schema mismatch; missing={missing}, extra={extra}")
    if frame.empty:
        raise ValueError("batch must contain at least one row")
    numeric = frame[NUMERIC_COLUMNS].apply(pd.to_numeric, errors="coerce")
    if numeric.isna().any().any():
        bad = numeric.columns[numeric.isna().any()].tolist()
        raise ValueError(f"non-numeric or missing values in required numeric columns: {bad}")
    if not frame["age"].between(18, 120).all():
        raise ValueError("age outside supported [18, 120] range")
    return frame[REQUIRED_COLUMNS].copy()

def predict_batch(frame, pipeline=final_pipeline, threshold=selected_threshold):
    valid = validate_batch(frame)
    probability = pipeline.predict_proba(valid)[:, 1]
    return pd.DataFrame({
        "subscription_probability": probability,
        "prioritize_call": probability >= threshold,
        "model_threshold": threshold,
    }, index=valid.index)

batch_result = predict_batch(X_val.head(5))
batch_result


,subscription_probability,prioritize_call,model_threshold
0,0.441084,True,0.22
1,0.055081,False,0.22
2,0.049041,False,0.22
3,0.034317,False,0.22
4,0.305101,True,0.22


## Serialization and reload verification


In [5]:
artifact_path = project_root() / "models" / "bank_marketing_pipeline.joblib"
metadata_path = project_root() / "reports" / "reproducibility_metadata.json"
joblib.dump({"pipeline": final_pipeline, "threshold": selected_threshold,
             "required_columns": REQUIRED_COLUMNS}, artifact_path)
reloaded = joblib.load(artifact_path)
before = final_pipeline.predict_proba(X_val.head(50))[:, 1]
after = reloaded["pipeline"].predict_proba(X_val.head(50))[:, 1]
np.testing.assert_allclose(before, after, rtol=0, atol=1e-12)

source_path = project_root() / "data" / "raw" / "bank-full.csv"
metadata = {
    **environment_metadata(), "seed": SEED, "fast_mode": FAST_MODE,
    "data_sha256": file_sha256(source_path), "threshold": selected_threshold,
    "development_rows": len(development), "validation_rows": len(validation),
    "test_rows": len(test), "final_test_metrics": final_test_metrics,
}
write_json(metadata, metadata_path)
print("reload predictions match; artifacts:", artifact_path, metadata_path)


reload predictions match; artifacts: /Users/soroush/Desktop/Personal/Projects/Daneshkar/advanced_machine_learning/models/bank_marketing_pipeline.joblib /Users/soroush/Desktop/Personal/Projects/Daneshkar/advanced_machine_learning/reports/reproducibility_metadata.json


## Basic unit-style checks and failure handling


In [6]:
assert "duration" not in REQUIRED_COLUMNS
assert set(predict_batch(X_val.head(3)).columns) == {
    "subscription_probability", "prioritize_call", "model_threshold"}
assert predict_batch(X_val.head(3))["subscription_probability"].between(0, 1).all()
try:
    predict_batch(X_val.head(1).drop(columns="age"))
    raise AssertionError("missing-column check did not fire")
except ValueError as exc:
    assert "missing=['age']" in str(exc)
try:
    invalid = X_val.head(1).copy(); invalid["age"] = 999
    predict_batch(invalid)
    raise AssertionError("range check did not fire")
except ValueError as exc:
    assert "age outside" in str(exc)
print("all unit-style checks passed")


all unit-style checks passed


## Model card

**Purpose:** prioritize clients immediately before a scheduled term-deposit marketing call.  
**Users:** authorized campaign analysts; not an autonomous eligibility or credit system.  
**Training data:** UCI Bank Marketing, Portuguese institution, historical campaigns; CC BY 4.0.  
**Target:** observed subscription after contact. This is prediction, not treatment effect.  
**Inputs:** 15 pre-contact fields; current-call `duration` is prohibited.  
**Model:** engineered mixed-type pipeline with one-hot encoding and LightGBM.  
**Threshold:** chosen on validation under illustrative cost `FP + 5 × FN`.  
**Evaluation:** test accessed once above; inspect the stored metrics in this executed notebook.  
**Limitations:** no customer ID or complete timestamp; possible repeated-client leakage; historical
and geographic transport risk; unknown fairness impact; probabilities may drift.  
**Out of scope:** causal targeting, credit decisions, fraud detection, or use outside governed campaigns.

## Monitoring plan

Log model/data versions, prediction ID, event time, schema outcomes, score, decision, and latency.
Immediately monitor volume, schema failures, unknown/missing rates, feature/prediction drift, and
decision rate. When labels arrive, join by prediction ID and monitor precision, recall, business cost,
calibration, and approved subgroup slices. Investigate sustained alerts before retraining; require
reproducible data, challenger validation, governance review, rollback, and shadow/canary checks.

## Deployment architecture (cloud-neutral)

A versioned training job emits the pipeline, threshold, metadata, and model card to an artifact
registry. A scheduled batch scorer validates an immutable input snapshot, writes versioned
predictions, and emits monitoring events. A later outcome job joins labels. Orchestration, secret
management, access control, audit logs, and rollback surround these components; a notebook is not
itself a production service.

## Common mistakes and leakage warnings

- Refitting on validation before test while retaining a threshold selected for the old model.
- Serializing only the estimator and reimplementing preprocessing in serving code.
- Accepting extra/missing columns silently.
- Treating a successful reload as proof of cross-version portability.
- Retraining on drift alerts without label-quality and performance investigation.

## Exercises

1. Add allowed-category checks with a policy for genuinely new categories.
2. Write a batch contract containing prediction IDs and event timestamps.
3. **Challenge:** package the scorer behind a local API with request validation, structured logs,
   concurrency limits, health checks, and a rollback test—without changing model semantics.

## Summary

The course closes with one artifact spanning deterministic data access, leakage-safe fitting,
validation-only threshold selection, one-time test evaluation, serialization, guarded inference,
reproducibility evidence, model documentation, and a delayed-label monitoring plan.

## References

- [scikit-learn model persistence](https://scikit-learn.org/stable/model_persistence.html)
- [scikit-learn pipelines](https://scikit-learn.org/stable/modules/compose.html#pipeline)
- [NIST AI Risk Management Framework](https://www.nist.gov/itl/ai-risk-management-framework)
- [Google model cards paper](https://doi.org/10.1145/3287560.3287596)
